In [ ]:
!pip install ultralytics
import zipfile
import os
import glob
import shutil
from sklearn.model_selection import train_test_split
import cv2
from ultralytics import YOLO
from IPython.display import Video, Image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 994.0/994.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [ ]:
zip_path = "/content/hands_dataset.zip"  # Replace with your actual file name
extract_path = "/content/hand_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# List all files to verify the dataset structure
for root, dirs, files in os.walk("/content/hand_dataset"):
    print(f"Folder: {root}")
    for file in files:
        print(f"  - {file}")

Folder: /content/hand_dataset
  - photo_2025-04-10_11-36-29.txt
  - photo_2025-04-10_11-36-36.txt
  - photo_2025-04-10_11-36-34.txt
  - photo_2025-04-10_11-36-33.jpg
  - hand9.txt
  - hand1.jpeg
  - hand1.txt
  - photo_2025-04-10_11-36-30.jpg
  - photo_2025-04-10_11-36-31.jpg
  - photo_2025-04-10_11-36-31.txt
  - photo_2025-04-10_11-36-34.jpg
  - photo_2025-04-10_11-36-33.txt
  - hand9.jpg
  - hand7.txt
  - photo_2025-04-10_11-36-35.jpg
  - photo_2025-04-10_11-36-29.jpg
  - photo_2025-04-10_11-36-37.txt
  - photo_2025-04-10_11-36-39.jpg
  - hand10.jpg
  - photo_2025-04-10_11-36-36.jpg
  - photo_2025-04-10_11-36-37.jpg
  - hand8.txt
  - photo_2025-04-10_11-36-39.txt
  - photo_2025-04-10_11-36-30.txt
  - photo_2025-04-10_11-36-35.txt
  - hand10.txt
  - hand7.jpg
  - hand8.jpg


In [ ]:
# Prepare image and label paths
images = glob.glob('/content/hand_dataset/*.jpg')
labels = glob.glob('/content/hand_dataset/*.txt')

# Pairing images with labels
image_basenames = [os.path.basename(f).replace('.jpg', '') for f in images]
label_basenames = [os.path.basename(f).replace('.txt', '') for f in labels]

# Split into train and val sets (80% train, 20% validation)
train_imgs, val_imgs = train_test_split(image_basenames, test_size=0.2, random_state=42)

# Make train and val folders
for split in ['train', 'val']:
    os.makedirs(f'/content/hand_dataset/images/{split}', exist_ok=True)
    os.makedirs(f'/content/hand_dataset/labels/{split}', exist_ok=True)

# Move images and labels to respective folders
for split, files in [('train', train_imgs), ('val', val_imgs)]:
    for base in files:
        shutil.move(f'/content/hand_dataset/{base}.jpg', f'/content/hand_dataset/images/{split}/{base}.jpg')
        shutil.move(f'/content/hand_dataset/{base}.txt', f'/content/hand_dataset/labels/{split}/{base}.txt')

In [ ]:
data_yaml = """
path: /content/hand_dataset
train: images/train
val: images/val
nc: 1
names: ['hand']
"""

with open('/content/hand_dataset/data.yaml', 'w') as f:
    f.write(data_yaml)

In [ ]:
model = YOLO('yolov8n.pt')  # Load YOLOv8n pretrained model
model.train(data='/content/hand_dataset/data.yaml', epochs=30)

100%|██████████| 6.25M/6.25M [00:00<00:00, 119MB/s]


Ultralytics 8.3.105 🚀 Python-3.11.11 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/hand_dataset/data.yaml, epochs=30, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_widt

100%|██████████| 755k/755k [00:00<00:00, 82.6MB/s]


Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

train: Scanning /content/hand_dataset/labels/train... 10 images, 0 backgrounds, 0 corrupt: 100%|██████████| 10/10 [00:00<00:00, 172.95it/s]

train: New cache created: /content/hand_dataset/labels/train.cache


... (93 outputs left out)


30 epochs completed in 0.083 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train/weights/best.pt, 6.2MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.105 🚀 Python-3.11.11 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  1.36it/s]


                   all          3          3    0.00333          1      0.995      0.962
Speed: 2.1ms preprocess, 211.6ms inference, 0.0ms loss, 5.9ms postprocess per image
Results saved to runs/detect/train


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d35b01d14d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
model = YOLO('yolov8n.pt')  # Load YOLOv8n pretrained model
model.train(data='/content/hand_dataset/data.yaml', epochs=30)

Ultralytics 8.3.105 🚀 Python-3.11.11 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/hand_dataset/data.yaml, epochs=30, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_wid

train: Scanning /content/hand_dataset/labels/train.cache... 10 images, 0 backgrounds, 0 corrupt: 100%|██████████| 10/10 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))



val: Scanning /content/hand_dataset/labels/val.cache... 3 images, 0 backgrounds, 0 corrupt: 100%|██████████| 3/3 [00:00<?, ?it/s]

Plotting labels to runs/detect/train2/labels.jpg... 


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs/detect/train2
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


... (89 outputs left out)


30 epochs completed in 0.083 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 6.2MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.105 🚀 Python-3.11.11 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  1.35it/s]


                   all          3          3    0.00333          1      0.995      0.962
Speed: 2.4ms preprocess, 208.5ms inference, 0.0ms loss, 5.4ms postprocess per image
Results saved to runs/detect/train2


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d35b4549dd0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
model = YOLO('runs/detect/train/weights/best.pt')  # Load the custom trained model

In [ ]:
results = model.predict(source='hand_video.mp4', save=True, conf=0.3)



WARNING ⚠️ inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/174) /content/hand_video.mp4: 640x384 (no detections), 170.1ms
video 1/1 (frame 2/174) /content/hand_video.mp4: 640x384 (no detections), 144.4ms
video 1/1 (frame 3/174) /content/hand_video.mp4: 640x384 (no detections), 143.7ms
video 1/1 (frame 4/174) /content/hand_video.mp4: 640x384 (no detections), 141.3ms
video 1/1 (frame 5/174) /content/hand_video.mp4: 640x384 (no detections), 142.5ms
video 1/1 (frame 6/174) /content/hand_video.mp

In [ ]:
Video("runs/detect/predict/hand_video.mp4", embed=True)  # Display the processed video

In [ ]:
# Print out confidence scores and class names
print(results[0].boxes)
print(results[0].names)
print(results[0].probs if hasattr(results[0], 'probs') else 'No probabilities')

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([])
conf: tensor([])
data: tensor([], size=(0, 6))
id: None
is_track: False
orig_shape: (854, 478)
shape: torch.Size([0, 6])
xywh: tensor([], size=(0, 4))
xywhn: tensor([], size=(0, 4))
xyxy: tensor([], size=(0, 4))
xyxyn: tensor([], size=(0, 4))
{0: 'hand'}
None
